In [1]:
from ultralytics import YOLO
import cv2



In [2]:

pose_model = YOLO("yolov8n-pose.pt")       # Pose model for shoulders
face_model = YOLO("yolov8n-face.pt")       # Face detection model

# ✅ Start Webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ✅ 1. Detect Pose (Shoulders for Heart)
    pose_results = pose_model(frame)

    for r in pose_results:
        keypoints = r.keypoints.xy.cpu().numpy()
        for person_kps in keypoints:
            left_shoulder  = person_kps[5]   # YOLO pose index 5
            right_shoulder = person_kps[6]   # YOLO pose index 6

            # ✅ Draw shoulder points
            cv2.circle(frame, tuple(map(int, left_shoulder)), 5, (0, 255, 0), -1)
            cv2.circle(frame, tuple(map(int, right_shoulder)), 5, (0, 255, 0), -1)

            # ✅ Midpoint between shoulders
            mid_x = int((left_shoulder[0] + right_shoulder[0]) / 2)
            mid_y = int((left_shoulder[1] + right_shoulder[1]) / 2)

            # ✅ Heart Location = Shoulder midpoint + downward offset
            heart_x = mid_x - 15    # little LEFT
            heart_y = mid_y + 40    # little DOWN
            cv2.circle(frame, (heart_x, heart_y), 10, (0, 0, 255), -1)
            cv2.putText(frame, "Heart",
                        (heart_x - 30, heart_y - 20),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (0, 0, 255), 2)

    # ✅ 2. Detect Face
    face_results = face_model(frame)
    for fr in face_results:
        for fbox in fr.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, fbox)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
            cv2.putText(frame, "Face",
                        (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (0, 255, 255), 2)

    # ✅ Show Frame
    cv2.imshow("YOLOv8 Pose + Face + Heart", frame)

    # ✅ Press 'Q' to Quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 1 person, 195.0ms
Speed: 2.6ms preprocess, 195.0ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 227.4ms
Speed: 2.1ms preprocess, 227.4ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 539.5ms
Speed: 6.6ms preprocess, 539.5ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 face, 597.2ms
Speed: 8.1ms preprocess, 597.2ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 613.0ms
Speed: 10.7ms preprocess, 613.0ms inference, 4.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 faces, 613.8ms
Speed: 25.3ms preprocess, 613.8ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 146.6ms
Speed: 2.3ms preprocess, 146.6ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 faces, 95.7ms
Speed: 2.1ms preprocess, 95.7ms inference, 1.1ms postprocess per image